# 🌿 Crop Disease Diagnostician — Full Pipeline

**Data download → Training → Convert → Evaluate → Download**

---

## ✅ Shuru karne se pehle:
1. **Runtime → Disconnect and delete runtime** (fresh start ke liye)
2. **Runtime → Change runtime type → T4 GPU → Save**
3. `kaggle.json` ready rakho ([kaggle.com](https://kaggle.com) → Profile → Settings → API → Create New Token)
4. **Runtime → Run All**

---
| Step | Time |
|---|---|
| Install + Data download | ~15 min |
| Model training | ~45 min |
| Convert + Evaluate + Download | ~15 min |

---
# 📦 STEP 1: Install Libraries
> ⚠️ `tensorflowjs` yahan install NAHI hoga — woh sirf conversion step mein install hoga (JAX conflict se bachne ke liye)

In [ ]:
# Only install packages that don't conflict with JAX/TF
# tensorflowjs is intentionally NOT installed here
!pip install -q kaggle seaborn scikit-learn
print('✅ Packages installed!')

---
# 🔑 STEP 2: Imports + Upload kaggle.json

In [ ]:
import os, json, shutil, random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from google.colab import files

print(f'✅ TensorFlow: {tf.__version__}')
print(f'✅ GPU: {tf.config.list_physical_devices("GPU")}')

tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('✅ Mixed precision: ON')

In [ ]:
print('File picker aayega → apna kaggle.json upload karo:')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ Kaggle ready!')

---
# 🌱 STEP 3: Download PlantVillage (~5 min)

In [ ]:
os.makedirs('/content/kaggle_download', exist_ok=True)
print('Downloading PlantVillage from Kaggle...')
!kaggle datasets download -d abdallahalidev/plantvillage-dataset \
    -p /content/kaggle_download --unzip

COLOR_DIR = None
for root, dirs, _ in os.walk('/content/kaggle_download'):
    if any('Tomato' in d for d in dirs) and any('Corn' in d for d in dirs):
        COLOR_DIR = root
        break

if COLOR_DIR is None:
    for c in ['/content/kaggle_download/plantvillage dataset/color',
              '/content/kaggle_download/color',
              '/content/kaggle_download/PlantVillage']:
        if os.path.exists(c):
            COLOR_DIR = c; break

assert COLOR_DIR, '❌ Dataset folder not found. Run: !find /content/kaggle_download -type d | head -20'
print(f'✅ Dataset at: {COLOR_DIR}')
print(f'Total classes: {len(os.listdir(COLOR_DIR))}')

---
# 🗂️ STEP 4: Organize 11 Classes → Train/Val/Test

In [ ]:
IMG_SIZE=224; BATCH_SIZE=32; SEED=42
DATA_DIR='/content/data'; MODEL_DIR='/content/model'
PHASE1_EPOCHS=5;  PHASE1_LR=1e-3
PHASE2_EPOCHS=15; PHASE2_LR=1e-4
UNFREEZE_FROM=100

os.makedirs(MODEL_DIR, exist_ok=True)
random.seed(SEED)

TARGET_CLASSES = {
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'corn_gray_leaf_spot',
    'Corn_(maize)___Common_rust_':                        'corn_common_rust',
    'Corn_(maize)___healthy':                             'corn_healthy',
    'Potato___Early_blight':                              'potato_early_blight',
    'Potato___Late_blight':                               'potato_late_blight',
    'Potato___healthy':                                   'potato_healthy',
    'Tomato___Bacterial_spot':                            'tomato_bacterial_spot',
    'Tomato___Early_blight':                              'tomato_early_blight',
    'Tomato___Late_blight':                               'tomato_late_blight',
    'Tomato___Leaf_Mold':                                 'tomato_leaf_mold',
    'Tomato___healthy':                                   'tomato_healthy',
}
LABELS_MAP  = sorted(TARGET_CLASSES.values())
NUM_CLASSES = len(LABELS_MAP)

print(f'{NUM_CLASSES} classes:')
for i, k in enumerate(LABELS_MAP): print(f'  [{i}] {k}')

print('\nFolder check:')
for f in TARGET_CLASSES:
    ok = os.path.exists(os.path.join(COLOR_DIR, f))
    print(f'  {"✅" if ok else "❌"} {f}')

In [ ]:
print('Organizing train/val/test (80/10/10)...')
split_stats = {}; total_train=total_val=total_test=0

for folder, class_key in TARGET_CLASSES.items():
    src = os.path.join(COLOR_DIR, folder)
    imgs = [f for f in os.listdir(src) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    random.shuffle(imgs)
    n=len(imgs); nv=max(1,int(n*.10)); nt=max(1,int(n*.10)); nr=n-nv-nt
    for sp, si in [('train',imgs[:nr]),('val',imgs[nr:nr+nv]),('test',imgs[nr+nv:])]:
        d=os.path.join(DATA_DIR,sp,class_key); os.makedirs(d,exist_ok=True)
        for img in si: shutil.copy(os.path.join(src,img), os.path.join(d,img))
    split_stats[class_key]={'train':nr,'val':nv,'test':nt}
    total_train+=nr; total_val+=nv; total_test+=nt
    print(f'  {class_key:<30} n={n} train={nr} val={nv} test={nt}')

print(f'\nTotal → train:{total_train} val:{total_val} test:{total_test}')

with open(f'{MODEL_DIR}/labels.json','w') as f: json.dump(LABELS_MAP,f,indent=2)

class_counts=[split_stats[k]['train'] for k in LABELS_MAP]
total_tr=sum(class_counts)
class_weights={i: total_tr/(NUM_CLASSES*c) for i,c in enumerate(class_counts)}

with open(f'{MODEL_DIR}/split_info.json','w') as f:
    json.dump({'labels':LABELS_MAP,'num_classes':NUM_CLASSES,
               'class_weights':{str(k):float(v) for k,v in class_weights.items()},
               'class_counts':class_counts}, f, indent=2)
print('✅ Data organized!')

---
# 🧠 STEP 5: Data Pipeline

In [ ]:
AUTOTUNE=tf.data.AUTOTUNE
assert sorted(os.listdir(f'{DATA_DIR}/train'))==LABELS_MAP, 'Class order mismatch!'
print('✅ Class order verified')

def make_dataset(split, augment=False):
    ds=tf.keras.utils.image_dataset_from_directory(
        f'{DATA_DIR}/{split}', image_size=(IMG_SIZE,IMG_SIZE),
        batch_size=None, label_mode='categorical',
        shuffle=(split=='train'), seed=SEED, class_names=LABELS_MAP)
    def preprocess(img, label):
        if augment:
            img=tf.image.random_flip_left_right(img)
            img=tf.image.random_flip_up_down(img)
            img=tf.image.random_brightness(img,0.2)
            img=tf.image.random_contrast(img,0.8,1.2)
            img=tf.image.random_saturation(img,0.8,1.2)
            img=tf.image.random_crop(img,[int(IMG_SIZE*.85),int(IMG_SIZE*.85),3])
            img=tf.image.resize(img,[IMG_SIZE,IMG_SIZE])
        return tf.keras.applications.mobilenet_v2.preprocess_input(img), label
    return ds.map(preprocess,num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

ds_train=make_dataset('train',augment=True)
ds_val  =make_dataset('val'  ,augment=False)
ds_test =make_dataset('test' ,augment=False)

imgs,lbls=next(iter(ds_train))
print(f'Batch: {imgs.shape}, {lbls.shape}')
print(f'Pixel range: [{imgs.numpy().min():.2f}, {imgs.numpy().max():.2f}]')
print('✅ Pipeline ready!')

---
# 🤖 STEP 6: Build MobileNetV2 Model

In [ ]:
def build_model(num_classes, trainable_base=False):
    base=tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE,IMG_SIZE,3), include_top=False, weights='imagenet')
    base.trainable=trainable_base
    inp=tf.keras.Input(shape=(IMG_SIZE,IMG_SIZE,3))
    x=base(inp,training=trainable_base)
    x=tf.keras.layers.GlobalAveragePooling2D()(x)
    x=tf.keras.layers.BatchNormalization()(x)
    x=tf.keras.layers.Dense(256,activation='relu')(x)
    x=tf.keras.layers.Dropout(0.35)(x)
    x=tf.cast(x,tf.float32)
    out=tf.keras.layers.Dense(num_classes,activation='softmax',dtype='float32')(x)
    return tf.keras.Model(inp,out), base

model, base_model = build_model(NUM_CLASSES)
print(f'Total params: {sum(tf.size(w).numpy() for w in model.weights):,}')
print('✅ Model built!')

---
# 🏋️ STEP 7: Phase 1 — Train Head (~10 min)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(PHASE1_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy'])

hist1=model.fit(
    ds_train, epochs=PHASE1_EPOCHS, validation_data=ds_val,
    class_weight=class_weights, verbose=1,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(f'{MODEL_DIR}/best_p1.keras',
            monitor='val_accuracy', save_best_only=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=2, verbose=1),
    ])
print(f'\n✅ Phase 1 done! Best val acc: {max(hist1.history["val_accuracy"])*100:.2f}%')

---
# 🔬 STEP 8: Phase 2 — Fine-Tune (~35 min)

In [ ]:
base_model.trainable=True
for l in base_model.layers[:UNFREEZE_FROM]: l.trainable=False
print(f'Unfrozen: {sum(1 for l in base_model.layers if l.trainable)} base layers')

model.compile(
    optimizer=tf.keras.optimizers.Adam(PHASE2_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy'])

hist2=model.fit(
    ds_train, initial_epoch=PHASE1_EPOCHS,
    epochs=PHASE1_EPOCHS+PHASE2_EPOCHS,
    validation_data=ds_val, class_weight=class_weights, verbose=1,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(f'{MODEL_DIR}/crop_disease_model.keras',
            monitor='val_accuracy', save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=2, verbose=1),
    ])

model.save(f'{MODEL_DIR}/crop_disease_model.h5')
print(f'\n✅ Training done! Best val acc: {max(hist2.history["val_accuracy"])*100:.2f}%')
print(f'Model saved: {MODEL_DIR}/crop_disease_model.h5')

In [ ]:
# Training curves
all_acc =hist1.history['accuracy']+hist2.history['accuracy']
all_vacc=hist1.history['val_accuracy']+hist2.history['val_accuracy']
all_loss=hist1.history['loss']+hist2.history['loss']
all_vloss=hist1.history['val_loss']+hist2.history['val_loss']

fig,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5))
for ax,tr,vl,title in [(ax1,all_acc,all_vacc,'Accuracy'),(ax2,all_loss,all_vloss,'Loss')]:
    ax.plot(tr,label='Train',color='#16a34a',lw=2)
    ax.plot(vl,label='Val',  color='#f59e0b',lw=2)
    ax.axvline(x=PHASE1_EPOCHS-.5,color='gray',ls='--',label='Fine-tune start')
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('Training Curves',fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/training_curves.png',dpi=120)
plt.show()
print('✅ Training curves saved!')

---
# 📊 STEP 9: Evaluate — Accuracy, F1, Confusion Matrix

In [ ]:
print('Evaluating on test set...')
y_true,y_pred=[],[]
for images,labels in ds_test:
    preds=model.predict(images,verbose=0)
    y_pred.extend(np.argmax(preds,axis=1).tolist())
    y_true.extend((np.argmax(labels.numpy(),axis=1) if len(labels.shape)>1 else labels.numpy()).tolist())
y_true=np.array(y_true); y_pred=np.array(y_pred)

top1_acc=accuracy_score(y_true,y_pred)
macro_f1=f1_score(y_true,y_pred,average='macro')

print(f'\n{"="*50}')
print(f'  ⭐  Resume mein ye numbers daalo!')
print(f'{"="*50}')
print(f'  Top-1 Accuracy : {top1_acc*100:.2f}%')
print(f'  Macro F1-Score : {macro_f1:.4f}')
print(f'  Test Samples   : {len(y_true)}')
print(f'{"="*50}')

report=classification_report(y_true,y_pred,target_names=LABELS_MAP,digits=4)
print(report)
with open(f'{MODEL_DIR}/classification_report.txt','w') as f:
    f.write(f'Accuracy:{top1_acc*100:.2f}% F1:{macro_f1:.4f}\n\n{report}')

In [ ]:
cm=confusion_matrix(y_true,y_pred)
cm_norm=cm.astype('float')/cm.sum(axis=1)[:,np.newaxis]
short=['C-GLS','C-Rst','C-H','P-EB','P-LB','P-H','T-BS','T-EB','T-LB','T-LM','T-H']
fig,ax=plt.subplots(figsize=(11,9))
sns.heatmap(cm_norm,annot=True,fmt='.2f',cmap='Greens',
            xticklabels=short,yticklabels=short,linewidths=.5,ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix (Acc:{top1_acc*100:.2f}%)',fontweight='bold')
plt.tight_layout(); plt.savefig(f'{MODEL_DIR}/confusion_matrix.png',dpi=150); plt.show()

per_f1=f1_score(y_true,y_pred,average=None)
colors=['#16a34a' if f>=.95 else '#f59e0b' if f>=.85 else '#ef4444' for f in per_f1]
fig,ax=plt.subplots(figsize=(10,5))
bars=ax.barh(LABELS_MAP,per_f1,color=colors)
ax.set_xlim(0,1.05); ax.axvline(x=macro_f1,color='gray',ls='--',label=f'Macro:{macro_f1:.3f}')
ax.set_title('Per-Class F1',fontweight='bold'); ax.legend()
for bar,f in zip(bars,per_f1):
    ax.text(bar.get_width()+.01,bar.get_y()+bar.get_height()/2,f'{f:.3f}',va='center',fontsize=9)
plt.tight_layout(); plt.savefig(f'{MODEL_DIR}/f1_per_class.png',dpi=120); plt.show()

with open(f'{MODEL_DIR}/metrics.json','w') as f:
    json.dump({'test_accuracy':round(float(top1_acc),4),'macro_f1':round(float(macro_f1),4),
               'test_samples':int(len(y_true)),
               'per_class_f1':{k:round(float(v),4) for k,v in zip(LABELS_MAP,per_f1)}},f,indent=2)
print('✅ Evaluation done!')

---
# ⚡ STEP 10: Convert to TFLite + TensorFlow.js
> `tensorflowjs` yahan install hota hai — training ke baad — isliye koi conflict nahi hoga

In [ ]:
# TFLite INT8 quantization (no tensorflowjs needed)
calib_ds=tf.keras.utils.image_dataset_from_directory(
    f'{DATA_DIR}/val', image_size=(IMG_SIZE,IMG_SIZE),
    batch_size=None, shuffle=True, seed=42, class_names=LABELS_MAP)

calib_imgs=list(calib_ds.take(200).map(
    lambda img,_: tf.expand_dims(
        tf.keras.applications.mobilenet_v2.preprocess_input(tf.cast(img,tf.float32)),0)))

def representative_dataset():
    for s in calib_imgs: yield [s]

TFLITE_PATH=f'{MODEL_DIR}/crop_disease_model.tflite'
conv=tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations=[tf.lite.Optimize.DEFAULT]
conv.representative_dataset=representative_dataset
conv.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8,tf.lite.OpsSet.TFLITE_BUILTINS]
conv.inference_input_type=tf.int8; conv.inference_output_type=tf.int8

print('Converting to INT8 TFLite...')
tflite=conv.convert()
with open(TFLITE_PATH,'wb') as f: f.write(tflite)

orig_mb=os.path.getsize(f'{MODEL_DIR}/crop_disease_model.h5')/1e6
tfl_mb=os.path.getsize(TFLITE_PATH)/1e6
print(f'✅ TFLite: {orig_mb:.1f}MB → {tfl_mb:.1f}MB ({orig_mb/tfl_mb:.1f}x smaller)')

In [ ]:
# NOW install tensorflowjs — training is already done so JAX conflict doesn't matter
print('Installing tensorflowjs for TFJS conversion...')
!pip install -q tensorflowjs
print('✅ tensorflowjs installed!')

In [ ]:
os.makedirs('/content/tfjs_model', exist_ok=True)
print('Converting to TensorFlow.js (float16)...')
!tensorflowjs_converter \
    --input_format=keras \
    --output_format=tfjs_layers_model \
    --quantize_float16 \
    /content/model/crop_disease_model.h5 \
    /content/tfjs_model/

shutil.copy(f'{MODEL_DIR}/labels.json', '/content/tfjs_model/labels.json')
tfjs_mb=sum(os.path.getsize(os.path.join('/content/tfjs_model',f))
            for f in os.listdir('/content/tfjs_model'))/1e6
print(f'✅ TFJS done! Size: {tfjs_mb:.1f} MB')
!ls -lh /content/tfjs_model/

---
# 📥 STEP 11: Download Everything

In [ ]:
shutil.make_archive('/content/tfjs_model_for_app','zip','/content/tfjs_model')
shutil.make_archive('/content/model_artifacts',   'zip', MODEL_DIR)

print('📦 Downloading 2 files:')
print('  1. tfjs_model_for_app.zip → unzip into app/public/model/')
print('  2. model_artifacts.zip    → TFLite + charts + report')

files.download('/content/tfjs_model_for_app.zip')
files.download('/content/model_artifacts.zip')

print(f'\n🎉 ALL DONE!')
print(f'   Accuracy : {top1_acc*100:.2f}%')
print(f'   F1 Score : {macro_f1:.4f}')
print()
print('Next steps:')
print('  1. Unzip tfjs_model_for_app.zip')
print('  2. Copy all files to: app/public/model/')
print('  3. cd app && npm run dev')
print('  4. Open http://localhost:5173')